## Jupyter themes

Install `jupyterthemes` with pip:

```
pip install jupyterthemes
```

You can get the list of available themes with:

```
jt -l
```

Assume we are going to install the theme `chesterish`, we can proceed by entering the following command:

```
jt -T -fs 14 -cellw 90% -t chesterish
```

- `-T` allows us to have toolbox,
- `-f` sets the font,
- `-fs` sets the font size,
- `-cellw` sets the cell width,
- `-t` sets the theme.

To restore the theme to defaut:

```
jt -r
```

## Installing third-party packages for spark

You can use the `spark-shell` command to install a package:

```
spark-shell --packages <package>
```

Pay attention to the messages while installing the package. If it is installed in `$SPARK_HOME/jars/`,
then after restarting the kernel, spark automatically detects the package. If the installation directory
is different form the above directory, then open the file `$SPARK_HOME/conf/spark-defaults.conf`
(if not exists, create it) and paste the following line into this file:

```
spark.jars path/*
```

where `path` is where the jar files of the package are installed.

## Apache toree, jupyter kernel for scala and spark

Install the kernel with:

```
pip install --upgrade toree
jupyter toree install --spark_home=/path/to/spark/directory --interpreters=Scala,SQL --user
```

## Setting number of partitions after shuffling tasks

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 2)

## Reading files into dataframe

Spark can read different file formats like:

- `csv`
- `parquet`
- `json`
- `orc`
- `com.databricks.spark.xml`
- `com.databricks.spark.avro`

into dataframes out of which the last two are not built-in spark formats.
Each format has its own option. For example, search
[here](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/index.html) for more information.

mode options:
- `permissive` [default]: for malformed rows, sets all the column values to NULL and push the entire row as a STRING
  to a column called `_corrupt_record`
- `dropmalformed`: drops the malformed rows
- `failfast`: raises exception

### CSV

In [ ]:
// Method 1

val df = spark.read.options(Map(
                                "header"->"true",
                                "inferSchema"->"true",
                                "nullValue"->"NA",
                                "timestampFormat"->"yyyy-MM-dd'T'HH:mm:ss",
                                "mode"->"failfast"
                                )
                           ).csv("/home/arian/Tutorials/survey.csv")

In [1]:
// Method 2

val df = spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .option("nullValue", "NA")
              .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
              .option("mode", "failfast")
              .option("path", "/home/arian/Tutorials/survey.csv")
              .load()

df = [Timestamp: string, Age: bigint ... 25 more fields]


[Timestamp: string, Age: bigint ... 25 more fields]

In [ ]:
// Method 3 (using a predefined schema)

import org.apache.spark.sql.types._

val mySchema = StructType(Array(
    StructField("timestamp",StringType,true),
    StructField("age",LongType,true),
    StructField("gender",StringType,true),
    StructField("country",StringType,true),
    StructField("state",StringType,true),
    StructField("self_employed",StringType,true),
    StructField("family_history",StringType,true),
    StructField("treatment",StringType,true),
    StructField("work_interfere",StringType,true),
    StructField("no_employees",StringType,true),
    StructField("remote_work",StringType,true),
    StructField("tech_company",StringType,true),
    StructField("benefits",StringType,true),
    StructField("care_options",StringType,true),
    StructField("wellness_program",StringType,true),
    StructField("seek_help",StringType,true),
    StructField("anonymity",StringType,true),
    StructField("leave",StringType,true),
    StructField("mental_health_consequence",StringType,true),
    StructField("phys_health_consequence",StringType,true),
    StructField("coworkers",StringType,true),
    StructField("supervisor",StringType,true),
    StructField("mental_health_interview",StringType,true),
    StructField("phys_health_interview",StringType,true),
    StructField("mental_vs_physical",StringType,true),
    StructField("obs_consequence",StringType,true),
    StructField("comments",StringType,true)))

val df = spark.read
              .format("csv")
              .schema(mySchema)
              .option("header", "true")
              .option("nullValue", "NA")
              .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
              .option("mode", "failfast")
              .option("path", "/home/arian/Tutorials/survey.csv")
              .load()

### parquet

In [ ]:
val df = spark.read
              .format("parquet")
              .option("option", "failfast")
              .load("/home/arian/Tutorials/parquet/")

### json

In [ ]:
val df = spark.read
              .format("json")
              .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
              .option("option", "failfast")
              .load("/home/arian/Tutorials/json/")

### ORC

In [ ]:
val df = spark.read
              .format("orc")
              .option("mode", "failfast")
              .load("/home/arian/Tutorials/orc/")

### XML

In [ ]:
val df = spark.read
              .format("com.databricks.spark.xml")
              .option("rowTag", "survey-row")
              .option("mode", "failfast")
              .load("/home/arian/Tutorials/xml/")

### avro

In [ ]:
val df = spark.read
              .format("avro")
              .option("mode", "failfast")
              .load("/home/arian/Tutorials/avro/")

In [2]:
//println(df.schema)
df.printSchema

root
 |-- age: long (nullable = true)
 |-- anonymity: string (nullable = true)
 |-- benefits: string (nullable = true)
 |-- care_options: string (nullable = true)
 |-- comments: string (nullable = true)
 |-- country: string (nullable = true)
 |-- coworkers: string (nullable = true)
 |-- family_history: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- leave: string (nullable = true)
 |-- mental_health_consequence: string (nullable = true)
 |-- mental_health_interview: string (nullable = true)
 |-- mental_vs_physical: string (nullable = true)
 |-- no_employees: string (nullable = true)
 |-- obs_consequence: string (nullable = true)
 |-- phys_health_consequence: string (nullable = true)
 |-- phys_health_interview: string (nullable = true)
 |-- remote_work: string (nullable = true)
 |-- seek_help: string (nullable = true)
 |-- self_employed: string (nullable = true)
 |-- state: string (nullable = true)
 |-- supervisor: string (nullable = true)
 |-- tech_company: string (

## Writing dataframes into files

mode options:
- `append`
- `overwrite`
- `errorIfExists`
- `ignore`

### CSV

In [3]:
df.write
  .format("csv")
  .option("header", "true")
  .option("nullValue", "NA")
  .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
  .mode("overwrite")
  .save("/home/arian/Tutorials/csv/")

### parquet

In [2]:
df.write
  .format("parquet")
  .mode("overwrite")
  .save("/home/arian/Tutorials/parquet/")

### json

In [3]:
df.write
  .format("json")
  .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
  .mode("overwrite")
  .save("/home/arian/Tutorials/json/")

### ORC

In [3]:
df.write
  .format("orc")
  .mode("overwrite")
  .save("/home/arian/Tutorials/orc/")

### XML

In [2]:
df.write
  .format("com.databricks.spark.xml")
  .option("rootTag", "survey")
  .option("rowTag", "survey-row")
  .mode("overwrite")
  .save("/home/arian/Tutorials/xml/")

### avro

In [2]:
df.write
  .format("avro")
  .mode("overwrite")
  .save("/home/arian/Tutorials/avro/")

## Partitioning a dataframe

In scala, dataframes are called and processed through the class *Dataset*. There is no separate class for dataframes.

In [ ]:
val dfPartitioned = df.repartition(5).toDF()
println(dfPartitioned.rdd.getNumPartitions)

The `toDF` method converts the strongly typed collection of data to generic Dataframe. In contrast to the strongly typed objects that Dataset operations work on, a Dataframe returns generic *Row* objects that allow fields to be accessed by ordinal or name. 

## Applying methods to the columns of a dataframe

In [ ]:
import org.apache.spark.sql.functions.{col,when}

val df1 = df.select(
    $"Gender", 
    when($"treatment"==="Yes", 1).otherwise(0) as "positive",
    when($"treatment"==="No", 1).otherwise(0) as "negative"
)
df1.show

In [ ]:
def parseGender(g: String) = {
    g.toLowerCase match {
    case "male" | "m" | "male-ish" | "maile" |
         "mal" |  "male (cis)" | "make" | "male " | 
         "man" | "msle" | "mail" | "malr" | 
         "cis man" | "cis male" => "Male"
    case "cis female" | "f" | "female" | "woman" |
         "femake" | "female " | "cis-female/femme" |
         "female (cis)" | "femail" => "Female"
    case _ => "Transgender"}
}

In [ ]:
// Make available the method for all the executors
import org.apache.spark.sql.functions.udf
val parseGenderUDF = udf(parseGender _)

In [ ]:
val df2 = df1.select(parseGenderUDF($"Gender") as "Gender", $"positive", $"negative")
df2.show

In [ ]:
/*
Creating a Relational Grouped Dataset by the column "Gender".
More precisely, thoes rows in df2 with the same Gender are
considered as a group.
*/

val RGD = df2.groupBy("Gender")

In [ ]:
// Calculate the summation over the columns "positive" and "negative" for each group


val df3 = RGD.sum("positive", "negative")
//val df3 = RGD.agg(Map("positive"->"sum", "negative"->"sum"))
df3.show

In [ ]:
// Counting the number of each group
RGD.count.show

In [ ]:
val df4 = df3.filter($"Gender" =!= "Transgender")
df4.collect

## Using SQL instead of dataframes

To do so, you should first convert the dataframe to a *view* using one of the following methods:

- `createGlobalTempView`
- `createOrReplaceGlobalTempView`
- `createTempView`
- `createOrReplaceTempView`

A view which is created using the first two methods is accessible from all the sessions running
inside the spark application, while, a view created by the last two methods is only accessible via
the session inside which it is created.

In [3]:
df.createOrReplaceTempView("myView")

// Global views are maintained inside "global_temp"
spark.catalog.listTables("global_temp").show

+------+--------+-----------+---------+-----------+
|  name|database|description|tableType|isTemporary|
+------+--------+-----------+---------+-----------+
|myview|    null|       null|TEMPORARY|       true|
+------+--------+-----------+---------+-----------+



Now, one can use SQL statements on `myView`. For example in the following box, the same task
as in the previous section is implemented on `myView` using SQL:

In [4]:
spark.sql("""select gender, sum(positive), sum(negative)
                from (select case when lower(trim(gender)) in ('male', 'm', 'male-ish', 'maile',
                                                               'mal',  'male (cis)', 'make', 'male ', 
                                                               'man', 'msle', 'mail', 'malr', 
                                                               'cis man', 'cis male')
                                  then 'Male'
                                  when lower(trim(gender)) in ('cis female', 'f', 'female', 'woman',
                                                            'femake', 'female ', 'cis-female/femme',
                                                            'female (cis)', 'femail')
                                  then 'Female'
                                  else 'Transgender'
                                  end as gender,
                             case when treatment == 'Yes' then 1 else 0 end as positive,
                             case when treatment ==  'No' then 1 else 0 end as negative
                      from myView)
                where gender != 'Transgender'
                group by gender""").show

+------+-------------+-------------+
|gender|sum(positive)|sum(negative)|
+------+-------------+-------------+
|Female|          170|           77|
|  Male|          450|          541|
+------+-------------+-------------+



To create a database

```
CREATE DATABASE mysparkdb
LOCATION '/home/arian/Tutorials/mysparkdb';
```

To get informaton about the database:

```
DESCRIBE DATABASE mysparkdb;
```

To get information about the **default** place where the databases are stored:

```
SET spark.sql.warehouse.dir
```

### HiveQL

Create a table using the command:

```
CREATE TABLE IF NOT EXISTS mysparkdb.hive_surveys(
TIME_STAMP TIMESTAMP,
AGE LONG,
GENDER STRING,
COUNTRY STRING,
STATE STRING,
SELF_EMPLOYED STRING,
FAMILY_HISTORY STRING,
TREATMENT STRING,
WORK_INTERFERE STRING,
NO_EMPLOYEES STRING,
REMOTE_WORK STRING,
TECH_COMPANY STRING,
BENEFITS STRING,
CARE_OPTIONS STRING,
WELLNESS_PROGRAM STRING,
SEEK_HELP STRING,
ANONYMITY STRING,
LEAVE STRING,
MENTAL_HEALTH_CONSEQUENCE STRING,
PHYS_HEALTH_CONSEQUENCE STRING,
COWORKERS STRING,
SUPERVISOR STRING,
MENTAL_HEALTH_INTERVIEW STRING,
PHYS_HEALTH_INTERVIEW STRING,
MENTAL_VS_PHYSICAL STRING,
OBS_CONSEQUENCE STRING,
COMMENTS STRING)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
STORED AS TEXTFILE;
```

And load the data into it:

```
LOAD DATA INPATH /home/arian/Tutorials/survey.csv'
INTO TABLE mysparkdb.hive_surveys;
```

**Note** that the tables loaded using HiveQL are not compatible with spark serialization,
hence, it will take a longer time to process such tables.

**Note** that the table loaded this way are MANAGED tables (a copy of the table exists in the database location).

### SparkSQL

SparkSQL performs well with spark serialization. It is recommended to use SparkSQL to load tables. <br>
To create a table using SparkSQL, replace:

```
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
STORED AS TEXTFILE;
```

by

```
USING CSV
OPTIONS (header='false',
         nullvalue='NA',
         timestampFormat='yyyy-MM-dd'T'HH:mm:ss');
```

You can replace `USING CSV` by `USING PARQUET` if you want to create the table in parquet format.

To load a table from an **external source** use the command:

```
LOCATION '/home/arian/Tutorials/survey.csv';
```

For deleting a table form a database, use the command:

```
DROP TABLE IF EXISTS mysparkdb.hive_surveys;
```

Creating a table from an existing one by selecting specific columns:

```
select TIME_STAM, AGE, GENDER, COUNTRY from mysparkdb.hive_surveys limit 5;
```

Get information about a table:

```
describe table extended mysparkdb.hive_surveys;
```

# Databricks

In [ ]:
// To remove a file on databricks
dbutils.fs.rm("/FileStore/tables/sample.txt")